In [45]:
import math
import os
import sys
from pathlib import Path

import einops
import numpy as np
import torch as t
from torch import Tensor

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part0_prereqs"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part0_prereqs.tests as tests
from part0_prereqs.utils import display_array_as_img, display_soln_array_as_img

MAIN = __name__ == "__main__"

## Einops

In [2]:
arr = np.load(section_dir / "numbers.npy")

In [3]:
print(arr[0].shape)
display_array_as_img(arr[0])

(3, 150, 150)


In [ ]:
print(arr[0, 0].shape)
display_array_as_img(arr[0, 0])

(150, 150)


In [5]:
arr_stacked = einops.rearrange(arr, "b c h w -> c h (b w)")
print(arr_stacked.shape)

(3, 150, 900)


In [6]:
display_array_as_img(arr_stacked)

In [7]:
arr1 = einops.rearrange(arr, "b c h w -> c (b h) w")
display_array_as_img(arr1)

In [8]:
arr2 = einops.repeat(arr[0], "c h w -> c (2 h) w")
display_array_as_img(arr2)

In [9]:
arr.shape

(6, 3, 150, 150)

In [10]:
arr[0:2].shape

(2, 3, 150, 150)

In [11]:
arr3 = einops.repeat(arr[0:2], "b c h w -> c (b h) (2 w)")
display_array_as_img(arr3)

In [12]:
arr4 = einops.repeat(arr[0], "c h w -> c (2 h) w")
display_array_as_img(arr4)

In [13]:
arr4 = einops.repeat(arr[0], "c h w -> c (h 2) w")
display_array_as_img(arr4)

In [14]:
arr5 = einops.rearrange(arr[0], "c h w -> h (c w)")
display_array_as_img(arr5)

In [15]:
arr5 = einops.rearrange(arr[0], "c h w -> (c h) w")
display_array_as_img(arr5)

In [16]:
arr6 = einops.rearrange(arr, "(b1 b2) c h w -> c (b1 h) (b2 w)", b1 = 2)
display_array_as_img(arr6)

In [17]:
arr7 = einops.rearrange(arr[1], "c h w -> c w h")
display_array_as_img(arr7)

In [18]:
arr8 = einops.reduce(arr, "(b1 b2) c (h h2) (w w2) -> c (b1 h) (b2 w)", "max", b1=2, h2=2, w2=2)
display_array_as_img(arr8)

## Broadcasting

**Rule 1**

If tensors don’t have same number of dimensions:
- Add leading 1s to the smaller one.


**Rule 2**

Then compare dimensions one by one:
- If equal → OK
- If one is 1 → stretch it
- Otherwise → ERROR

In [19]:
x = t.ones((3, 1, 5))
y = t.ones((1, 4, 5))

z = x + y

In [20]:
x, y

(tensor([[[1., 1., 1., 1., 1.]],
 
         [[1., 1., 1., 1., 1.]],
 
         [[1., 1., 1., 1., 1.]]]),
 tensor([[[1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.]]]))

In [21]:
z

tensor([[[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]],

        [[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]],

        [[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]]])

In [22]:
z.shape

torch.Size([3, 4, 5])

In [23]:
x = t.ones((8, 2, 6))
y = t.ones((2, 6))      # rule 1: y.shape (1, 2, 6)  # rule 2: y.shape (8, 2, 6)

# this is valid
z = x + y

In [24]:
z.shape

torch.Size([8, 2, 6])

In [25]:
x = t.ones((10, 20, 30))
y = t.ones((20, 1))     

# rule 1: append a single dimension to the front of y: (1, 20, 1)
# rule 2: comparing dimenstions: - dim0 in y can be streched to match x, - dim1 is equal in both, - dim2 can be streched to match x
#         so y is now (10, 20, 30) which will be the shape of z

# so this is valid
z = x + y
z.shape

torch.Size([10, 20, 30])

In [26]:
x = t.ones((4, 1))
y = t.ones((4,))

x, y
# I think this will work without doing anything? or do I need to to transpos one?
# No, actually y's dimension was expanded (1, 4) then bothe were stretched to (4, 4)
z = x + y

In [27]:
z, z.shape

(tensor([[2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.]]),
 torch.Size([4, 4]))

In [28]:
x = t.ones((3, 1, 5))

print(x.unsqueeze(3).shape) # (3, 1, 5, 1) because we add a new dummy dimension at idx 3 (the end) in the new tensot
print(x.squeeze(1).shape) # (3, 5) because we remove the dimension at idx 1
print(x.squeeze(0).shape) # (3, 1, 5) because we don't remove the leading dimension (it has size 3)

torch.Size([3, 1, 5, 1])
torch.Size([3, 5])
torch.Size([3, 1, 5])


In [29]:
x = t.ones((1, 4, 6))

print(x.squeeze(0).shape) # (4, 6)

torch.Size([4, 6])


In [30]:
def assert_all_equal(actual: Tensor, expected: Tensor) -> None:
    assert actual.shape == expected.shape, f"Shape mismatch, got: {actual.shape}"
    assert (actual == expected).all(), f"Value mismatch, got {actual}"
    print("Tests passed!")

def assert_all_close(actual: Tensor, expected: Tensor, atol=1e-3) -> None:
    assert actual.shape == expected.shape, f"Shape mismatch, got: {actual.shape}"
    t.testing.assert_close(actual, expected, atol=atol, rtol=0.0)
    print("Tests passed!")

#### (A1) rearrange

In [31]:
def rearrange_1() -> Tensor:
    """Return the following tensor using only t.arange and einops.rearrange:
    
    [[3, 4],
     [5, 6],
     [7, 8]]
     """
    x = t.arange(3,9)
    x = einops.rearrange(x, '(b1 b2) -> b1 b2', b1=3)
    return x

In [32]:
expected = t.tensor([[3, 4], [5, 6], [7, 8]])
assert_all_equal(rearrange_1(), expected)

Tests passed!


The parentheses on the left:
```
(h w)
```

Mean:

"This single input dimension is actually composed of two logical dimensions multiplied together."

That's all.

#### (A2) rearrange

In [33]:
def rearrange_2() -> Tensor:
    """Return the following tensor using only t.arange and einops.rearrange:

    [[1, 2, 3],
     [4, 5, 6]]
    """

    return einops.rearrange(t.arange(1, 7), '(h w) -> h w', h=2)

In [34]:
assert_all_equal(rearrange_2(), t.tensor([[1, 2, 3], [4, 5, 6]]))

Tests passed!


#### (B1) temperature average

In [35]:
def temperature_average(temps: Tensor) -> Tensor:
    """Return the average temperature for each week.
    
    temps: a 1D temperature contining temperature for each day.
    Length will be a multiple of 7 and the first 7 days are for the first week, second 7 days for the second week, etc
    
    You can do with a single call to reduce.
    """

    assert len(temps) % 7 == 0
    return einops.reduce(temps, '(b n) -> b', 'mean', n=7)


In [36]:
temps = t.tensor([71, 72, 70, 75, 71, 72, 70, 75, 80, 85, 80, 78, 72, 83]).float()
expected = [71.571, 79.0]
assert_all_close(temperature_average(temps), t.tensor(expected))

Tests passed!


#### (B2) temperature difference

In [37]:
temps = t.tensor([71, 72, 70, 75, 71, 72, 70, 75, 80, 85, 80, 78, 72, 83]).float()
avgs = temperature_average(temps)
avgs

tensor([71.5714, 79.0000])

In [38]:
temps = einops.rearrange(temps, '(week day) -> day week', day=7)

In [39]:
t = temps - avgs
t

tensor([[-0.5714, -4.0000],
        [ 0.4286,  1.0000],
        [-1.5714,  6.0000],
        [ 3.4286,  1.0000],
        [-0.5714, -1.0000],
        [ 0.4286, -7.0000],
        [-1.5714,  4.0000]])

In [40]:
t = einops.rearrange(t, 'day week -> week day')
t

tensor([[-0.5714,  0.4286, -1.5714,  3.4286, -0.5714,  0.4286, -1.5714],
        [-4.0000,  1.0000,  6.0000,  1.0000, -1.0000, -7.0000,  4.0000]])

In [41]:
t.flatten()

tensor([-0.5714,  0.4286, -1.5714,  3.4286, -0.5714,  0.4286, -1.5714, -4.0000,
         1.0000,  6.0000,  1.0000, -1.0000, -7.0000,  4.0000])

In [42]:
einops.repeat(avgs, "w -> (w 7)")

tensor([71.5714, 71.5714, 71.5714, 71.5714, 71.5714, 71.5714, 71.5714, 79.0000,
        79.0000, 79.0000, 79.0000, 79.0000, 79.0000, 79.0000])

In [43]:
def temperatures_differences(temps: Tensor) -> Tensor:
    """For each day, subtract the average for the week the day belongs to.
    
    temps: a 1D temperature containing temperatures for each day.
    Length will be a multiple of 7 and the first 7 days are for the first week, second 7 days for the second week, etc.

    """
    assert len(temps) % 7 == 0
    avgs = temperature_average(temps)
    temps = einops.rearrange(temps, '(week day) -> day week', day=7)
    result = einops.rearrange((temps - avgs), 'day week -> week day')
    return result.flatten()

In [46]:
temps = t.tensor([71, 72, 70, 75, 71, 72, 70, 75, 80, 85, 80, 78, 72, 83]).float()
expected = [-0.571, 0.429, -1.571, 3.429, -0.571, 0.429, -1.571, -4.0, 1.0, 6.0, 1.0, -1.0, -7.0, 4.0]
actual = temperatures_differences(temps)
assert_all_close(actual, t.tensor(expected))

Tests passed!


#### (B3) temperature normalized

In [47]:
temps = t.tensor([71, 72, 70, 75, 71, 72, 70, 75, 80, 85, 80, 78, 72, 83]).float()
avgs = einops.reduce(temps, '(week days) -> week', 'mean', days=7)
avgs

tensor([71.5714, 79.0000])

In [48]:
avgs = einops.repeat(avgs, 'avg_per_week -> (avg_per_week day)', day=7)
avgs

tensor([71.5714, 71.5714, 71.5714, 71.5714, 71.5714, 71.5714, 71.5714, 79.0000,
        79.0000, 79.0000, 79.0000, 79.0000, 79.0000, 79.0000])

In [49]:
diffs = temps - avgs
diffs

tensor([-0.5714,  0.4286, -1.5714,  3.4286, -0.5714,  0.4286, -1.5714, -4.0000,
         1.0000,  6.0000,  1.0000, -1.0000, -7.0000,  4.0000])

In [50]:
std = einops.reduce(temps, '(week days) -> week', t.std, days=7)
std

tensor([1.7182, 4.4721])

In [51]:
std = einops.repeat(std, 'weekly_std -> (weekly_std day)', day=7)
std

tensor([1.7182, 1.7182, 1.7182, 1.7182, 1.7182, 1.7182, 1.7182, 4.4721, 4.4721,
        4.4721, 4.4721, 4.4721, 4.4721, 4.4721])

In [52]:
diffs/std

tensor([-0.3326,  0.2494, -0.9146,  1.9954, -0.3326,  0.2494, -0.9146, -0.8944,
         0.2236,  1.3416,  0.2236, -0.2236, -1.5652,  0.8944])

In [53]:
def temperatures_normalized(temps: Tensor) -> Tensor:
    """For each day, subtract the weekly average and divide by the weekly standard deviation.
    
    temps: as above"""
    assert len(temps) % 7 == 0
    avg = einops.reduce(temps, '(week days) -> week', 'mean', days=7)
    std = einops.reduce(temps, '(week days) -> week', t.std, days=7)
    diffs = temps - einops.repeat(avg, 'w -> (w d)', d=7)
    return diffs/einops.repeat(std, 'w -> (w d)', d=7)

In [54]:
expected = [-0.333, 0.249, -0.915, 1.995, -0.333, 0.249, -0.915, -0.894, 0.224, 1.342, 0.224, -0.224, -1.565, 0.894]
actual = temperatures_normalized(temps)
assert_all_close(actual, t.tensor(expected))

Tests passed!


#### (C1) normalize a matrix

In [55]:
matrix = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).float()
matrix

tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])

In [56]:
# this is a norm of the entire tensor/matrix
norm1 = matrix.norm()
norm1, norm1.shape

(tensor(16.8819), torch.Size([]))

In [57]:
# this is a row-wise norm w/o keepdim
norm2 = matrix.norm(dim=1)
norm2, norm2.shape
# So we lost the column dimension.

(tensor([ 3.7417,  8.7750, 13.9284]), torch.Size([3]))

In [58]:
matrix.shape

torch.Size([3, 3])

In [59]:
# this is a row-wise norm with keepdim
norm3 = matrix.norm(dim=1, keepdim=True)
norm3, norm3.shape

(tensor([[ 3.7417],
         [ 8.7750],
         [13.9284]]),
 torch.Size([3, 1]))

In [60]:
def normalize_rows(matrix: Tensor) -> Tensor:
    """Normalize each row of the given 2D matrix.
    
    matrix: a 2D tensor of shape (m, n)
    
    Returns: a tensor of the same shape where each row is divided by its l2 norm
    """
    return (matrix / matrix.norm(dim=1, keepdim=True))

In [61]:
matrix = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).float()
expected = t.tensor([[0.267, 0.535, 0.802], [0.456, 0.570, 0.684], [0.503, 0.574, 0.646]])
assert_all_close(normalize_rows(matrix), expected)

Tests passed!


#### (C2) pairwise cosine similarity

In [62]:
def cos_sim_matrix(matrix: Tensor) -> Tensor:
    """Return the cosine similarity matrix for each pair of rows of the given matrix.
    
    matrix: shape (m, n)
    """
    X_norm = matrix / matrix.norm(dim=1, keepdim=True)
    return X_norm @ X_norm.T

In [63]:
matrix = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).float()
expected = t.tensor([[1.0, 0.975, 0.959], [0.975, 1.0, 0.998], [0.959, 0.998, 1.0]])
assert_all_close(cos_sim_matrix(matrix), expected)

Tests passed!


#### (D) sample distribution

In [67]:
def sample_distribution(probs: Tensor, n: int) -> Tensor:
    """Return n random samples from probs, where probs is a normalized probability distribution.
    
    probs: shape (k,) where probs[i] is the probability of event i occuring.
    n: number of random samples
    
    Return: shape (n,) where out[i] is an integer indicating which even was sampled.
    """
    cdf = t.cumsum(probs, dim=0)
    random_samples = t.rand(n)
    return (random_samples.unsqueeze(1) >= cdf.unsqueeze(0)).sum(dim=1)

In [68]:
n = 5_000_000
probs = t.tensor([0.05, 0.1, 0.1, 0.2, 0.15, 0.4])
freqs = t.bincount(sample_distribution(probs, n)) / n
assert_all_close(freqs, probs)

Tests passed!


(E) classifier accuracy

In [69]:
def classifier_accuracy(scores: Tensor, true_classes: Tensor) -> Tensor:
    """
    Return the fraction of inputs for which the maximum score corresponds to the true class for that input.
    
    scores: shape (batch, n_classess). A higher score[b, i] means that the classifer thinks that class i is more likely.
    true_classes: shape (batch, ). true_classes[b] is an integer from [0...n_classes]
    """
    correct = (t.argmax(scores, dim=1) == true_classes).sum()
    total = true_classes.size(0)
    return correct/total

In [70]:
scores = t.tensor([[0.75, 0.5, 0.25], [0.1, 0.5, 0.4], [0.1, 0.7, 0.2]])
true_classes = t.tensor([0, 1, 0])
expected = 2.0 / 3.0
assert classifier_accuracy(scores, true_classes) == expected
print("Tests passed!")

Tests passed!


(F1) total price indexing

In [71]:
def total_price_indexing(prices: Tensor, items: Tensor) -> float:
    """Given prices for each kind of item and a tensor of items purchased, return the total price.
    
    prices: shape(k, ). prices[i] is the price of the ith item.
    items: shap(n, ). a 1D tensor where each value is an item index from [0..k).
    """
    return prices[items].sum().item()

In [72]:
prices = t.tensor([0.5, 1, 1.5, 2, 2.5])
items = t.tensor([0, 0, 1, 1, 4, 3, 2])
assert total_price_indexing(prices, items) == 9.0
print("Tests passed!")

Tests passed!


In [73]:
prices[items].sum().item()

9.0

(F2) gather 2D

In [74]:
def gather_2d(matrix: Tensor, indexes: Tensor) -> Tensor:
    """Perform a gather operation along the second dimension.
    
    matrix: shape(m, n)
    indexes: shape (m, k)
    
    Return: shape (m, k). out[i][j] = matrix[i][index[i][j]]"""
    assert matrix.ndim == indexes.ndim # make sure we have the sam number of dims
    assert indexes.shape[0] == matrix.shape[0] # since we're gathering along dim=1, rows must align
    assert indexes.max() < matrix.shape[1] # make sure no invalid indices exist

    out = matrix.gather(1, indexes)
    assert out.shape == indexes.shape   # make sure the out shape it what is expected

    return out

In [75]:
matrix = t.arange(15).view(3, 5)
indexes = t.tensor([[4], [3], [2]])
expected = t.tensor([[4], [8], [12]])
assert_all_equal(gather_2d(matrix, indexes), expected)

indexes2 = t.tensor([[2, 4], [1, 3], [0, 2]])
expected2 = t.tensor([[2, 4], [6, 8], [10, 12]])
assert_all_equal(gather_2d(matrix, indexes2), expected2)

Tests passed!
Tests passed!


(F3) total price gather

In [76]:
def total_price_gather(prices: Tensor, items: Tensor) -> float:
    """compute the same as total_price_indexing, but use torch.gather."""
    assert items.max() < prices.shape[0]
    return t.gather(prices, 0, items).sum().item()

In [77]:
prices = t.tensor([0.5, 1, 1.5, 2, 2.5])
items = t.tensor([0, 0, 1, 1, 4, 3, 2])
assert total_price_gather(prices, items) == 9.0
print("Tests passed!")

Tests passed!


(G) indexing

In [78]:
def integer_array_indexing(matrix: Tensor, coords: Tensor) -> Tensor:
    """Return the values at each coodinate using integer arry indexing.
    
    matrix: shape (d_0, d_1, ..., d_n)
    coords: shape (batch, n)
    
    Return: (batch, )
    """
    assert matrix.ndim == coords.shape[1]
    idx = [coords[:,d] for d in range(coords.shape[1])]
    return matrix[idx]
    

In [79]:
mat_2d = t.arange(15).view(3, 5)
coords_2d = t.tensor([[0, 1], [0, 4], [1, 4]])
actual = integer_array_indexing(mat_2d, coords_2d)
assert_all_equal(actual, t.tensor([1, 4, 9]))

mat_3d = t.arange(2 * 3 * 4).view((2, 3, 4))
coords_3d = t.tensor([[0, 0, 0], [0, 1, 1], [0, 2, 2], [1, 0, 3], [1, 2, 0]])
actual = integer_array_indexing(mat_3d, coords_3d)
assert_all_equal(actual, t.tensor([0, 5, 10, 15, 20]))

Tests passed!
Tests passed!


/tmp/ipykernel_8689/2093696607.py:11: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return matrix[idx]


In [80]:
tuple(coords_2d.T)

(tensor([0, 0, 1]), tensor([1, 4, 4]))

In [81]:
mat_2d[tuple(coords_2d.T)]

tensor([1, 4, 9])

In [82]:
coords_3d.T

tensor([[0, 0, 0, 1, 1],
        [0, 1, 2, 0, 2],
        [0, 1, 2, 3, 0]])

In [83]:
mat_3d[tuple(coords_3d.T)]

tensor([ 0,  5, 10, 15, 20])

(H1) batched logssumexp

In [84]:
def batched_logsumexp(matrix: Tensor) -> Tensor:
    """For each row of the matrix, compute log(sum(exp(row))) in a numerically stable way.
    
    matrix: shape (batch, n)
    
    Return: (bathc, ). For each i, out[i] = log(sum(exp(matrix[i])))
    """
    c = matrix.max(dim=1).values
    c = c.unsqueeze(1)
    return (c.squeeze(-1) + t.log(t.sum(t.exp(matrix - c), dim=1)))

In [86]:
matrix = t.tensor([[-1000, -1000, -1000, -1000], [1000, 1000, 1000, 1000]])
expected = t.tensor([-1000 + math.log(4), 1000 + math.log(4)])
actual = batched_logsumexp(matrix)
assert_all_close(actual, expected)

Tests passed!


In [87]:
matrix2 = t.randn((10, 20))
expected2 = t.logsumexp(matrix2, dim=-1)
actual2 = batched_logsumexp(matrix2)
assert_all_close(actual2, expected2)

Tests passed!


(H2) batched softmax

In [88]:
def batched_softmax(matrix: Tensor) -> Tensor:
    """For each row of the matrix, compute softmax(row).
    
    matrix: shape (batch, n)
    
    Return: (batch, n). For each i, out[i] should sum to 1"""
    # softmax(score) = e^(score)/ sum(e^(score))
    exp = matrix.exp()
    return exp / exp.sum(dim=-1, keepdim=True)

In [89]:
matrix = t.arange(1, 6).view((1, 5)).float().log()
expected = t.arange(1, 6).view((1, 5)) / 15.0
actual = batched_softmax(matrix)
assert_all_close(actual, expected)
for i in [0.12, 3.4, -5, 6.7]:
    assert_all_close(actual, batched_softmax(matrix + i))

Tests passed!
Tests passed!
Tests passed!
Tests passed!
Tests passed!


In [90]:
matrix2 = t.rand((10, 20))
actual2 = batched_softmax(matrix2)
assert actual2.min() >= 0.0
assert actual2.max() <= 1.0
assert_all_equal(actual2.argsort(), matrix2.argsort())
assert_all_close(actual2.sum(dim=-1), t.ones(matrix2.shape[:-1]))

Tests passed!
Tests passed!


(H3) batched logsoftmax

In [91]:
def batched_logsoftmax(matrix: Tensor) -> Tensor:
    """Compute log(softmax(row)) for each row of the matrix.
    
    matrix: shape (batch, n)
    
    Return: (batch, n).
    
    For each row, subtract the maximum first to avoid overflow if the row contains large values."""
    C = matrix.max(dim=-1).values
    C = C.unsqueeze(1)
    exps = t.exp(matrix - C)
    softmax = exps / t.sum(exps, dim=-1, keepdim=True)
    return t.log(softmax)

In [93]:
matrix = t.arange(1, 7).view((2, 3)).float()
start = 1000
matrix2 = t.arange(start + 1, start + 7).view((2, 3)).float()
actual = batched_logsoftmax(matrix2)
expected = t.tensor([[-2.4076, -1.4076, -0.4076],
                     [-2.4076, -1.4076, -0.4076]])
assert_all_close(actual, expected)

Tests passed!


(H4) batched cross entropy loss

In [94]:
def batched_cross_entropy_loss(logits: Tensor, true_labels: Tensor) -> Tensor:
    """Compute the cross entropy loss for each example in the batch.
    
    logits: shape (batch, classes), logit[i][j] is the unnormalized prediction for example i and class j.
    true_labels: shape (batch, ). true_labels[i] is an integer index representing the true class for example i.
    
    Return: shape (batch, ). out[i] is the loss for example i.
    
    hint: convert the logits to log-probabilities using you batched_logsoftmax from above.
    Then the loss for an example is just the negative of the log-probability that the model assigned to the true class.
    Use torch.gather to perform the indexing.
    """
    assert logits.shape[0] == true_labels.shape[0]
    assert true_labels.max() < logits.shape[1]
    log_probs = batched_logsoftmax(logits)
    return (-log_probs.gather(1, true_labels.unsqueeze(1))).squeeze(-1)

In [96]:
logits = t.tensor([[float("-inf"), float("-inf"), 0], [1 / 3, 1 / 3, 1 / 3], [float("-inf"), 0, 0]])
true_labels = t.tensor([2, 0, 0])
expected = t.tensor([0.0, math.log(3), float("inf")])
actual = batched_cross_entropy_loss(logits, true_labels)
assert_all_close(actual, expected)

Tests passed!


(I1) collect rows

In [97]:
def collect_rows(matrix: Tensor, row_indexes: Tensor) -> Tensor:
    """Reurn a 2D matrix whose rows are taken from the input matrix in order according to row_indexes.

     matrix: shape (m, n)
     row_indexes: shape (k,), Each value is an integer in [0..m).
     
     Return: shape (k, n), out[i] is matrix[row_indexes[i]]
     """
    assert row_indexes.max() < matrix.shape[0]
    return matrix[row_indexes]

In [98]:
matrix = t.arange(15).view((5, 3))
row_indexes = t.tensor([0, 2, 1, 0])
actual = collect_rows(matrix, row_indexes)
expected = t.tensor([[0, 1, 2], [6, 7, 8], [3, 4, 5], [0, 1, 2]])
assert_all_equal(actual, expected)

Tests passed!


(I2) collect columns

In [99]:
def collect_columns(matrix: Tensor, column_indexes: Tensor) -> Tensor:
    """return a 2D matrix whose columns are taken from the input matrix in order according to column_indexes.
    
    matrix: shape (m, n)
    column_indexes: shape (k,). Each value is an intefer in [0..n).
    
    Return: shape (m, k), out [:, i] is matrix[:, column_indexes[i]].
    """
    assert column_indexes.max() < matrix.shape[1]
    return matrix[:, column_indexes]

In [100]:
matrix = t.arange(15).view((5, 3))
column_indexes = t.tensor([0, 2, 1, 0])
actual = collect_columns(matrix, column_indexes)
expected = t.tensor([[0, 2, 1, 0], [3, 5, 4, 3], [6, 8, 7, 6], [9, 11, 10, 9], [12, 14, 13, 12]])
assert_all_equal(actual, expected)

Tests passed!


## Einsum

In [101]:
def einsum_trace(mat: np.ndarray):
    """
    Returns the same as `np.trace`.
    """
    return einops.einsum(mat, "i i ->")

def einsum_mv(mat: np.ndarray, vec: np.ndarray):
    """
    Returns the same as `np.matmul`, when `mat` is a 2D array and `vec` is 1D.
    """
    return einops.einsum(mat, vec, "i j, j -> i")

def einsum_mm(mat1: np.ndarray, mat2: np.ndarray):
    """
    Returns the same as `np.matmul` when `mat1` and `mat2` are both 2D arrays.
    """
    return einops.einsum(mat1, mat2, "i j, j k -> i k")

def einsum_inner(vec1: np.ndarray, vec2: np.ndarray):
    """
    Returns the sam as `np.inner`.
    """
    return einops.einsum(vec1, vec2, "i, i ->")

def einsum_outer(vec1: np.ndarray, vec2: np.ndarray):
    """
    Returns the same as `np.outer`.
    """
    return einops.einsum(vec1, vec2, "i, j -> i j")

In [102]:
tests.test_einsum_trace(einsum_trace)
tests.test_einsum_mv(einsum_mv)
tests.test_einsum_mm(einsum_mm)
tests.test_einsum_inner(einsum_inner)
tests.test_einsum_outer(einsum_outer)

All tests in `test_einsum_trace` passed!
All tests in `test_einsum_mv` passed!
All tests in `test_einsum_mm` passed!
All tests in `test_einsum_inner` passed!
All tests in `test_einsum_outer` passed!
